# Profilage de la charge et des pannes d'un réseau électrique régional avec PROC CHART

## Résumé

Un analyste des opérations du réseau utilise PROC CHART pour profiler un échantillon synthétique de relevés de compteurs de circuits de distribution à travers quatre régions de service et quatre sources de production. Le notebook parcourt des diagrammes à barres verticales, à barres horizontales, en blocs et en secteurs pour résumer le mix de production, comparer la charge moyenne et totale par région, mettre en évidence le pic de demande du soir par heure et classer les minutes de panne par source — le type d'exploration rapide, à sortie texte, qu'une équipe du secteur de l'énergie et des services publics mène avant de s'engager dans un rapport graphique. L'étape DATA demande 1 200 lignes ; ce notebook a été rendu en mode sans licence, qui plafonne le jeu de données de travail aux 100 premiers relevés, de sorte que chaque diagramme ci-dessous résume cet échantillon de 100 lignes.

## Sources de données

| Jeu de données | Lignes | Description |
|---------|------|-------------|
| `grid_ops` | 100 (échantillon synthétique) | Une ligne par relevé de compteur de circuit de distribution sur un réseau régional, généré en ligne avec `call streaminit(20260531)` et `rand()`. La boucle de l'étape DATA demande 1 200 relevés, mais le mode sans licence plafonne le jeu de données enregistré à 100 observations, de sorte que les diagrammes ci-dessous décrivent ces 100 lignes. |

# Profilage de la charge et des pannes d'un réseau électrique régional avec PROC CHART

PROC CHART produit des diagrammes à barres, en blocs et en secteurs en mode caractère directement dans la sortie listing — sans nécessiter de périphérique ODS Graphics. Cela en fait un outil d'exploration rapide de première passe pour une équipe des opérations du réseau qui veut *voir* la forme de ses données de charge et de fiabilité avant de construire des visuels soignés GCHART ou SGPLOT.

Dans ce notebook, nous :

1. Générons un mois synthétique de relevés de compteurs de circuits de distribution pour un exploitant de réseau régional.
2. Représentons le **mix de production** (part des relevés par source).
3. Comparons la **charge moyenne et totale livrée** entre les régions de service.
4. Mettons en évidence le **pic de demande du soir** par heure de la journée.
5. Utilisons un **diagramme en blocs** pour croiser région et source de production.
6. Classons les **minutes de panne** par source pour trouver l'alimentation la moins fiable.

Chaque instruction et option ci-dessous relève de la syntaxe standard SAS 9.4 de PROC CHART.

## Étape 1 — Générer les données d'exploitation synthétiques

L'étape DATA ci-dessous fabrique des relevés de compteurs dans une boucle de 1 200 itérations. Chaque relevé se voit attribuer une région de service, une source de production (le gaz domine, l'éolien, le solaire et l'hydraulique se partageant le reste) et une heure de la journée. La charge est plus élevée dans la fenêtre du soir 17h00–21h00 (et nous marquons ces relevés comme pic), et nous tirons les minutes de panne d'une loi de Poisson. `call streaminit` fixe la graine aléatoire afin que les données soient reproductibles.

Le NOTE dans le journal montre le résultat pratique : cette exécution est en mode sans licence, de sorte que le jeu de données `grid_ops` enregistré est plafonné aux 100 premiers de ces relevés (100 lignes, 7 colonnes). Chaque diagramme qui suit résume cet échantillon de 100 lignes, et chaque journal de PROC CHART confirme qu'il a lu 100 lignes.

In [1]:
/* Synthetic feeder-circuit operations for a regional grid operator */
DONNÉES grid_ops;
    APPELER streaminit(20260531);
    LONGUEUR region $12 source $9 peak_flag $3;
    TABLEAU regions[4] $12 _temporary_
        ('North','South','East','West');
    FAIRE meter_id = 1 JUSQU_À 1200;
        r = ceil(4 * rand('uniform'));
        region = regions[r];
        u = rand('uniform');
        SI u < 0.42 ALORS source = 'Gas';
        SINON SI u < 0.70 ALORS source = 'Wind';
        SINON SI u < 0.88 ALORS source = 'Solar';
        SINON source = 'Hydro';
        hour = floor(24 * rand('uniform'));
        BASE = 18 + 5 * (region = 'North') + 3 * (region = 'West');
        SI hour >= 17 AND hour <= 21 ALORS FAIRE;
            load_mwh = BASE + 12 + 6 * rand('normal');
            peak_flag = 'Yes';
        FIN;
        SINON FAIRE;
            load_mwh = BASE + 4 * rand('normal');
            peak_flag = 'No';
        FIN;
        SI load_mwh < 0 ALORS load_mwh = 0;
        outage_min = rand('poisson', 2.5);
        SORTIE;
    FIN;
    SUPPRIMER r u BASE;
EXÉCUTER;

NOTE: DATA grid_ops

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote grid_ops (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds


## Étape 2 — Mix de production

Quelle part des relevés chaque source de production représente-t-elle ? Un diagramme à barres verticales avec `TYPE=PERCENT` y répond directement : la hauteur des barres correspond au pourcentage de toutes les observations tombant dans chaque catégorie de source. Comme `source` est une variable caractère, PROC CHART traite automatiquement ses valeurs comme des catégories discrètes.

In [2]:
PROCÉDURE chart DONNÉES=grid_ops;
    VBAR source / type=PERCENT;
EXÉCUTER;


Percent of SOURCE

         |              *****       
         |              *****       
   40.00 +              *****       
         |              *****       
         |              *****       
   30.00 +              *****       
         |       *****  *****       
         |       *****  *****       
   20.00 +       *****  *****       
         |       *****  *****       
         |*****  *****  *****       
         |*****  *****  *****  *****
   10.00 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          Hydro  Wind    Gas   Solar
                    SOURCE


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Étape 3 — Charge moyenne livrée par région

Nous passons maintenant du comptage au résumé d'une mesure. Nommer `load_mwh` dans `SUMVAR=` avec `TYPE=MEAN` fait de la longueur de la barre la charge moyenne par région, et nous demandons explicitement les colonnes de statistiques : `MEAN` imprime la moyenne à côté de chaque barre et `CFREQ` ajoute une colonne de fréquence cumulée. La région Nord porte la charge moyenne livrée la plus élevée (moyenne d'environ 28,0 MWh), cohérente avec le décalage régional intégré aux données, tandis que le Sud est le plus bas (environ 19,8 MWh).

Nous passons aussi `ASCENDING`, dans l'intention d'ordonner les barres de la moyenne la plus faible à la plus élevée. Notez dans la sortie que les barres ne sont **pas** réordonnées — elles apparaissent dans l'ordre des catégories (Ouest, Sud, Est, Nord), avec des moyennes de 24,2, 19,8, 21,7 et 28,0 qui ne vont pas de la plus courte à la plus longue. Le réordonnancement des barres selon la statistique du diagramme n'est pas encore implémenté dans cette version de PROC CHART, de sorte que `ASCENDING`/`DESCENDING` sont acceptés mais restent actuellement sans effet ; lisez plutôt le classement dans la colonne `Mean` imprimée.

In [3]:
PROCÉDURE chart DONNÉES=grid_ops;
    HBAR region / SUMVAR=load_mwh type=mean
                  cfreq mean ascending;
EXÉCUTER;


Mean of REGION

REGION                                                  Mean           N     Percent
                                                                                    
  West  **********************************             24.17       24.17       23.00
 South  ****************************                   19.80       43.97       21.00
  East  *******************************                21.72       65.69       29.00
 North  ****************************************       28.03       93.72       27.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Étape 4 — Charge totale par région

Ici, `TYPE=SUM` fait de chaque barre la charge livrée *totale* de la région plutôt que la moyenne, de sorte que les barres les plus hautes marquent les régions livrant le plus d'énergie agrégée sur l'ensemble des relevés échantillonnés. Le Nord domine à nouveau pour la charge totale livrée.

L'instruction demande également `SUBGROUP=peak_flag`, demandant à PROC CHART de scinder chaque barre selon que le relevé tombait ou non dans la fenêtre de pic du soir. Dans SAS, cela segmente chaque barre avec un glyphe distinct par niveau de sous-groupe et imprime une légende. Cette version ne rend pas encore la segmentation par sous-groupe (une lacune de capacité documentée), de sorte que les barres apparaissent comme des suites pleines de `*` sans décomposition par segment — les *totaux* des barres sont corrects, mais la répartition pic/hors-pic n'est pas montrée ici. Pour voir quelle quantité de charge tombe en heures de pic, utilisez la vue par heure de la journée de l'étape 5.

In [4]:
PROCÉDURE chart DONNÉES=grid_ops;
    VBAR region / SUMVAR=load_mwh type=sum
                  SUBGROUP=peak_flag;
EXÉCUTER;


Sum of REGION

         |                     *****
         |                     *****
         |                     *****
     600 +              *****  *****
         |*****         *****  *****
         |*****         *****  *****
         |*****         *****  *****
     400 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
     200 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          West   South  East   North
                    REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Étape 5 — Forme de la charge au fil de la journée

`hour` est continue, aussi fixons-nous nous-mêmes le regroupement avec une liste `MIDPOINTS=` explicite à des centres espacés de 4 heures (2, 6, 10, 14, 18, 22). Chaque barre montre la charge moyenne livrée pour les relevés proches de cette heure. La barre centrée sur 18 devrait ressortir — c'est le pic de demande du soir que nous avons intégré aux données.

In [5]:
PROCÉDURE chart DONNÉES=grid_ops;
    VBAR hour / SUMVAR=load_mwh type=mean
                midpoints=2 6 10 14 18 22;
EXÉCUTER;


Mean of HOUR

         |                            *****       
         |                            *****       
   30.00 +                            *****       
         |                            *****  *****
         |                            *****  *****
         |              *****         *****  *****
   20.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
   10.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |                                        
         +----------------------------------------+
            2      6     10     14     18     22  
                            HOUR


NOTE: PROC CHART data=grid_ops


## Étape 6 — Région par source de production (diagramme en blocs)

Un diagramme BLOCK rend un petit tableau à deux entrées sous forme de grille de blocs. Avec `GROUP=source` et `SUMVAR=load_mwh / TYPE=MEAN`, le diagramme croise chaque région avec la source de production qui la dessert, la hauteur des blocs étant proportionnelle à la charge moyenne — une manière compacte de repérer quelles combinaisons région/source portent la charge moyenne la plus lourde.

In [6]:
PROCÉDURE chart DONNÉES=grid_ops;
    BLOCK region / SUMVAR=load_mwh type=mean
                   GROUPE=source;
EXÉCUTER;


Block chart of Mean of REGION by SOURCE

                          /####\
  /####\                  /####\
  /####\          /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  +----+  +----+  +----+  +----+
   West   South    East   North 
             REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Étape 7 — Mix de production sous forme de diagramme en secteurs

La même information de part par source de l'étape 2, dessinée en secteurs. PIE avec `TYPE=PERCENT` dimensionne chaque secteur selon son pourcentage du total des relevés et imprime une légende des pourcentages de secteurs à côté de la figure.

In [7]:
PROCÉDURE chart DONNÉES=grid_ops;
    PIE source / type=PERCENT;
EXÉCUTER;


Pie chart of SOURCE

          SOURCE     Percent   Percent  Slice
----------------  ----------  --------  ------------------------------
           Hydro       14.00     14.0%  ####
            Wind       28.00     28.0%  ********
             Gas       45.00     45.0%  ++++++++++++++
           Solar       13.00     13.0%  OOOO


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Étape 8 — Minutes de panne par source

Enfin, nous classons la fiabilité. `SUMVAR=outage_min` avec `TYPE=SUM` totalise les minutes de panne par source. Nous passons `DESCENDING` pour tenter de faire remonter la source la moins performante en tête, mais comme à l'étape 3, les barres ne sont pas réordonnées — elles s'impriment dans l'ordre des catégories (Hydraulique, Éolien, Gaz, Solaire) et le réordonnancement des barres n'est pas encore implémenté. Lisez le classement dans la colonne `Sum` imprimée : le gaz représente le plus grand total de minutes de panne dans cet échantillon (122), bien devant l'éolien (64), le solaire (43) et l'hydraulique (38). Cela suit le mix de production — le gaz dessert le plus de relevés, il accumule donc le plus de minutes de panne au total.

In [8]:
PROCÉDURE chart DONNÉES=grid_ops;
    HBAR source / SUMVAR=outage_min type=sum DESCENDANT;
EXÉCUTER;


Sum of SOURCE

SOURCE                                                   Sum        Cum.     Percent
                                                                     Sum            
 Hydro  ************                                      38          38       14.00
  Wind  *********************                             64         102       28.00
   Gas  ****************************************         122         224       45.00
 Solar  **************                                    43         267       13.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Interprétation des résultats

La lecture conjointe des diagrammes donne à l'équipe des opérations une image situationnelle rapide (sur l'échantillon de 100 lignes capturé par cette exécution) :

- **Mix de production (étapes 2 et 7).** Le gaz porte la plus grande part des relevés (environ 45 %), l'éolien en deuxième (environ 28 %) et l'hydraulique et le solaire en retrait (environ 14 % et 13 %) — la barre verticale et le secteur racontent la même histoire de deux façons, une vérification de cohérence utile.
- **Charge par région (étapes 3 et 4).** Le HBAR de charge moyenne montre le Nord avec la charge moyenne livrée la plus élevée (moyenne ~28 MWh) et le Sud la plus basse (~20 MWh), cohérent avec le décalage régional intégré aux données. Le diagramme SUM confirme que le Nord livre aussi le plus d'énergie totale.
- **Forme de la charge quotidienne (étape 5).** La barre de point médian centrée sur l'heure 18 est nettement la plus haute, confirmant le pic de demande 17h00–21h00 que nous avons intégré aux données — précisément là où un service public concentrerait la réponse à la demande et la planification de capacité.
- **Fiabilité (étape 8).** Le total des minutes de panne par source fait ressortir le gaz comme le plus grand contributeur d'indisponibilité dans cet échantillon (122 minutes), la cible naturelle suivante pour une revue de maintenance — bien que cela reflète surtout le fait que le gaz dessert le plus de relevés.

Deux options d'affichage utilisées ici — le réordonnancement des barres `ASCENDING`/`DESCENDING` (étapes 3 et 8) et la segmentation des barres `SUBGROUP=` (étape 4) — sont acceptées par l'analyseur mais ne sont pas encore rendues par cette version de PROC CHART, de sorte que les classements et les répartitions se lisent dans les colonnes de statistiques imprimées plutôt que dans l'ordre ou l'ombrage des barres.

PROC CHART ne produit que de la sortie caractère, aussi, pour des visuels prêts pour un conseil d'administration, l'équipe porterait ces mêmes vues vers PROC GCHART ou PROC SGPLOT. Mais comme première passe sans configuration sur un extrait frais, ces diagrammes répondent aux questions opérationnelles — mix, ampleur et calendrier — en quelques secondes.